# Embeddings Explained — From Words to Vectors to Retrieval

A hands-on walkthrough of:
1. What embeddings actually are
2. How to visualise them
3. How to train a lightweight embedding model from scratch
4. How to build a simple retrieval system with cosine similarity

No heavy dependencies — everything runs on CPU with standard packages.

## 1. Import Required Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity
import torch
import torch.nn as nn
import torch.optim as optim
from collections import defaultdict

# Reproducibility
np.random.seed(42)
torch.manual_seed(42)

print("All libraries loaded successfully.")

## 2. What Are Embeddings? (Code Demo)

An **embedding** converts a discrete token (word, sentence, ID) into a dense vector of real numbers.

Think of it like a map: every word gets a GPS coordinate. Words with similar meanings land near each other.

The simplest embedding is a **lookup table** — each word has its own row of learned numbers.

In [ ]:
# --- Manual lookup-table embedding (no ML needed) ---
# Vocabulary: a tiny Healthcare-flavoured word list
vocab = [
    "patient", "doctor", "nurse", "hospital", "clinic",
    "prescription", "diagnosis", "surgery", "cancer", "diabetes",
    "appointment", "medication", "blood", "heart", "lung",
]

# Create a mapping from words to indices
word2idx = {w: i for i, w in enumerate(vocab)}

VOCAB_SIZE = len(vocab)
EMBED_DIM  = 8  # small dimension so we can print it

# Random initialisation — these would be *learned* during training
embedding_matrix = np.random.randn(VOCAB_SIZE, EMBED_DIM).astype(np.float32)

## Function to get embedding for a word
def get_embedding(word):
    """Return the embedding vector for a single word."""
    idx = word2idx.get(word)
    if idx is None:
        raise ValueError(f"'{word}' not in vocab")
    return embedding_matrix[idx]

# Demo: look up a few words
for word in ["patient", "doctor", "hospital"]:
    vec = get_embedding(word)
    print(f"{word:15s} → {np.round(vec, 2)}")

## 3. Visualising Word Embeddings in 2D

Raw embeddings live in high-dimensional space (8-D above, 768-D for BERT).  
We squash them to 2-D with **PCA** or **t-SNE** so we can plot them.

- **PCA** — fast, linear, great for an overview  
- **t-SNE** — slower, non-linear, better at revealing clusters

In [ ]:
# --- Craft hand-tuned embeddings so clusters are visible (simulating trained embeddings) ---
# Group 1: healthcare workers  (centred around  [2, 2])
# Group 2: conditions          (centred around [-2, 2])
# Group 3: clinical actions    (centred around  [0,-2])

np.random.seed(0)
groups = {
    "Healthcare workers": (["patient", "doctor", "nurse"], [2.0,  2.0]),
    "Conditions":         (["cancer", "diabetes", "heart", "lung"], [-2.0,  2.0]),
    "Clinical actions":   (["surgery", "diagnosis", "prescription", "medication", "appointment", "blood"], [0.0, -2.0]),
    "Facilities":         (["hospital", "clinic"], [3.0, -1.0]),
}

# Build a 2-D embedding for illustration
illustrative_2d = {}
for group_name, (words, center) in groups.items():
    for w in words:
        illustrative_2d[w] = np.array(center) + np.random.randn(2) * 0.4

# --- PCA on the 8-D random embeddings (real pipeline) ---
pca = PCA(n_components=2)
reduced = pca.fit_transform(embedding_matrix)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: PCA on random (untrained) embeddings — no structure expected
ax = axes[0]
ax.scatter(reduced[:, 0], reduced[:, 1], c="steelblue", s=80, alpha=0.7)
for i, word in enumerate(vocab):
    ax.annotate(word, (reduced[i, 0], reduced[i, 1]), fontsize=8, ha="right")
ax.set_title("PCA of random (untrained) embeddings\n→ no structure yet", fontsize=11)
ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2")

# Right: hand-crafted 2-D embeddings showing semantic clusters
ax = axes[1]
colors = {"Healthcare workers": "#e07b39", "Conditions": "#4c7fb0",
          "Clinical actions": "#56a055", "Facilities": "#9b59b6"}
for group_name, (words, _) in groups.items():
    xs = [illustrative_2d[w][0] for w in words]
    ys = [illustrative_2d[w][1] for w in words]
    ax.scatter(xs, ys, label=group_name, color=colors[group_name], s=100, alpha=0.85)
    for w, x, y in zip(words, xs, ys):
        ax.annotate(w, (x, y), fontsize=8, ha="right")
ax.legend(fontsize=8)
ax.set_title("Simulated *trained* embeddings\n→ semantic clusters emerge", fontsize=11)
ax.set_xlabel("Dim 1"); ax.set_ylabel("Dim 2")

plt.tight_layout()
plt.show()
print("Explained variance by PCA:", np.round(pca.explained_variance_ratio_, 3))

## 4. Building a Lightweight Embedding Model

We build a tiny PyTorch model:

```
Input word index  →  Embedding layer (lookup)  →  Linear projection  →  L2-normalised vector
```

The **embedding layer** is just a learnable matrix.  
The **projection head** maps it to a fixed output size and normalises the vector so cosine similarity is well-behaved.

In [ ]:
class LightweightEmbedder(nn.Module):
    """
    Word-level embedding model.
      vocab_size  : number of unique tokens
      embed_dim   : internal embedding dimension
      output_dim  : final projected dimension (what we use for retrieval)
    """
    def __init__(self, vocab_size: int, embed_dim: int = 16, output_dim: int = 8):
        super().__init__()
        self.embedding  = nn.Embedding(vocab_size, embed_dim)
        self.projection = nn.Sequential(
            nn.Linear(embed_dim, output_dim),
            nn.Tanh(),
        )

    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        # token_ids: (batch,)  →  embeds: (batch, embed_dim)
        embeds = self.embedding(token_ids)
        projected = self.projection(embeds)           # (batch, output_dim)
        # L2 normalise so all vectors sit on the unit sphere
        normed = projected / (projected.norm(dim=-1, keepdim=True) + 1e-8)
        return normed


model = LightweightEmbedder(vocab_size=VOCAB_SIZE, embed_dim=16, output_dim=8)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

## 5. Training the Embedding Model

We use a **contrastive (triplet-style) objective**:

> Pull *similar* word pairs closer together, push *dissimilar* pairs further apart.

**Dataset**: hand-labelled pairs of (anchor, positive, negative) from our Healthcare vocabulary.

**Loss**: `max(0,  d(anchor, positive) - d(anchor, negative) + margin)`  
where d is the **cosine distance** (1 − cosine similarity).

In [ ]:
# --- Triplet dataset: (anchor, positive, negative) ---
# Similar pairs share a semantic group; negative is from a different group
triplets_raw = [
    # anchor          positive        negative
    ("patient",       "doctor",       "cancer"),
    ("doctor",        "nurse",        "surgery"),
    ("nurse",         "patient",      "medication"),
    ("hospital",      "clinic",       "diabetes"),
    ("cancer",        "diabetes",     "hospital"),
    ("diabetes",      "heart",        "nurse"),
    ("heart",         "lung",         "appointment"),
    ("surgery",       "diagnosis",    "patient"),
    ("prescription",  "medication",   "doctor"),
    ("appointment",   "diagnosis",    "lung"),
    ("blood",         "heart",        "clinic"),
    ("medication",    "prescription", "hospital"),
]

# Convert words → indices
triplets = [
    (word2idx[a], word2idx[p], word2idx[n])
    for a, p, n in triplets_raw
]

# --- Training loop ---
EPOCHS  = 300
MARGIN  = 0.3
LR      = 0.02

optimizer = optim.Adam(model.parameters(), lr=LR)
losses = []

for epoch in range(EPOCHS):
    epoch_loss = 0.0
    for a_idx, p_idx, n_idx in triplets:
        a = torch.tensor([a_idx])
        p = torch.tensor([p_idx])
        n = torch.tensor([n_idx])

        v_a = model(a)   # (1, 8)
        v_p = model(p)
        v_n = model(n)

        # Cosine similarity in [-1, 1]; cosine distance = 1 - similarity
        sim_ap = (v_a * v_p).sum()          # scalar
        sim_an = (v_a * v_n).sum()
        dist_ap = 1.0 - sim_ap
        dist_an = 1.0 - sim_an

        loss = torch.clamp(dist_ap - dist_an + MARGIN, min=0.0)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    losses.append(epoch_loss / len(triplets))

# --- Plot training loss ---
plt.figure(figsize=(8, 3))
plt.plot(losses, color="steelblue", linewidth=1.5)
plt.xlabel("Epoch"); plt.ylabel("Avg triplet loss")
plt.title("Training loss — embedding model")
plt.tight_layout()
plt.show()
print(f"Final loss: {losses[-1]:.4f}")

## 6. Vectorising Text Queries

Once trained, converting any word in the vocabulary to a retrieval-ready vector is a single forward pass.

In [ ]:
model.eval()

def vectorise(word: str) -> np.ndarray:
    """Return the trained L2-normalised embedding for a word."""
    idx = torch.tensor([word2idx[word]])
    with torch.no_grad():
        vec = model(idx)
    return vec.squeeze().numpy()

# Vectorise the entire vocabulary to build our document store
doc_embeddings = {word: vectorise(word) for word in vocab}

# Example: vectorise a query
query_word = "patient"
query_vec  = vectorise(query_word)
print(f"Query: '{query_word}'")
print(f"Vector (dim={len(query_vec)}): {np.round(query_vec, 3)}")
print(f"L2 norm: {np.linalg.norm(query_vec):.4f}  (should be ≈ 1.0)")

## 7. Cosine Similarity — Formula & Implementation

Cosine similarity measures the **angle** between two vectors, not their length:

$$\cos(\theta) = \frac{\mathbf{A} \cdot \mathbf{B}}{\|\mathbf{A}\| \, \|\mathbf{B}\|}$$

- Result in **[−1, 1]**: `1` = identical direction, `0` = orthogonal, `−1` = opposite  
- Because our vectors are L2-normalised (∥v∥ = 1), the formula simplifies to the **dot product**: $\cos(\theta) = \mathbf{A} \cdot \mathbf{B}$

In [ ]:
def cosine_sim_numpy(a: np.ndarray, b: np.ndarray) -> float:
    """Cosine similarity from scratch."""
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8))

# --- Verify against sklearn ---
a = vectorise("patient")
b = vectorise("doctor")
c = vectorise("cancer")

sim_ab_manual  = cosine_sim_numpy(a, b)
sim_ab_sklearn = cosine_similarity(a.reshape(1, -1), b.reshape(1, -1))[0, 0]

sim_ac_manual  = cosine_sim_numpy(a, c)

print("Cosine similarities (manual vs sklearn)")
print(f"  patient ↔ doctor  : manual={sim_ab_manual:.4f}, sklearn={sim_ab_sklearn:.4f}")
print(f"  patient ↔ cancer  : manual={sim_ac_manual:.4f}")
print()
print("Interpretation:")
print(f"  patient↔doctor ({sim_ab_manual:.3f}) should be > patient↔cancer ({sim_ac_manual:.3f})")
print(f"  Training pulled similar words together: {'✓' if sim_ab_manual > sim_ac_manual else '✗'}")

## 8. Building a Simple Retrieval System

A **vector store** in its simplest form is just a dictionary of `{doc_id: vector}`.

Retrieval steps:
1. Vectorise the query → `q`
2. Compute `cosine_similarity(q, doc_i)` for every document
3. Rank by similarity, return top-k

In [ ]:
class SimpleVectorStore:
    """Minimal in-memory vector store."""

    def __init__(self):
        self.store: dict[str, np.ndarray] = {}

    def add(self, doc_id: str, vector: np.ndarray):
        self.store[doc_id] = vector

    def query(self, query_vec: np.ndarray, top_k: int = 5):
        """Return top_k (doc_id, similarity) pairs sorted descending."""
        scores = [
            (doc_id, cosine_sim_numpy(query_vec, vec))
            for doc_id, vec in self.store.items()
        ]
        scores.sort(key=lambda x: x[1], reverse=True)
        return scores[:top_k]


# --- Build the vector store from our vocabulary ---
store = SimpleVectorStore()
for word, vec in doc_embeddings.items():
    store.add(word, vec)

# --- Run retrieval for several queries ---
queries = ["patient", "surgery", "cancer"]

for q_word in queries:
    q_vec   = vectorise(q_word)
    results = store.query(q_vec, top_k=5)
    print(f"\nQuery: '{q_word}'")
    print(f"  {'Rank':<6} {'Word':<18} {'Cosine sim':>10}")
    print(f"  {'-'*36}")
    for rank, (word, sim) in enumerate(results, 1):
        marker = " ← query" if word == q_word else ""
        print(f"  {rank:<6} {word:<18} {sim:>10.4f}{marker}")

## 9. Visualising Query vs Document Vectors

We project all trained vectors to 2-D with PCA, then highlight a query and its nearest neighbours.  
This gives a visual confirmation that retrieval is doing the right thing — close vectors in 2-D are close in the original 8-D space too.

In [ ]:
def plot_retrieval(query_word: str, top_k: int = 4):
    """
    Project trained embeddings to 2-D and visualise retrieval results.
    The query is shown as a star; top-k neighbours are highlighted in orange;
    all other documents are grey.
    """
    all_words = list(doc_embeddings.keys())
    all_vecs  = np.stack([doc_embeddings[w] for w in all_words])  # (15, 8)

    # PCA to 2-D
    pca2 = PCA(n_components=2)
    coords = pca2.fit_transform(all_vecs)           # (15, 2)
    word2coord = {w: coords[i] for i, w in enumerate(all_words)}

    # Retrieval
    q_vec    = vectorise(query_word)
    top_hits = store.query(q_vec, top_k=top_k + 1)           # +1 to skip the query itself
    top_hits = [(w, s) for w, s in top_hits if w != query_word][:top_k]
    hit_words = {w for w, _ in top_hits}

    fig, ax = plt.subplots(figsize=(8, 6))

    # Draw faint lines from query to each retrieved doc
    q_coord = word2coord[query_word]
    for hit_word, sim in top_hits:
        h_coord = word2coord[hit_word]
        ax.plot(
            [q_coord[0], h_coord[0]], [q_coord[1], h_coord[1]],
            color="orange", linewidth=1.2, linestyle="--", alpha=0.6,
            zorder=1,
        )
        # Annotate midpoint with similarity score
        mid = ((q_coord[0] + h_coord[0]) / 2, (q_coord[1] + h_coord[1]) / 2)
        ax.text(mid[0], mid[1], f"{sim:.2f}", fontsize=7, color="darkorange", ha="center")

    # Plot all documents
    for word in all_words:
        x, y = word2coord[word]
        if word == query_word:
            ax.scatter(x, y, c="crimson", s=200, marker="*", zorder=5)
            ax.annotate(f"  ★ {word} (query)", (x, y), fontsize=9,
                        color="crimson", fontweight="bold")
        elif word in hit_words:
            rank = [w for w, _ in top_hits].index(word) + 1
            ax.scatter(x, y, c="darkorange", s=100, zorder=4)
            ax.annotate(f"  #{rank} {word}", (x, y), fontsize=8, color="darkorange")
        else:
            ax.scatter(x, y, c="lightgrey", s=60, zorder=3)
            ax.annotate(f"  {word}", (x, y), fontsize=7, color="grey")

    ax.set_title(
        f"Retrieval for query='{query_word}'  |  top-{top_k} neighbours highlighted",
        fontsize=11,
    )
    ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2")
    plt.tight_layout()
    plt.show()


# --- Run the visualisation for three different queries ---
for q in ["patient", "surgery", "cancer"]:
    plot_retrieval(q, top_k=4)

## Cosine Similarity Heatmap — All Pairs

A final overview: the cosine similarity matrix across every word pair.  
After training, words in the same semantic group should show warmer colours.

In [ ]:
all_vecs_mat = np.stack([doc_embeddings[w] for w in vocab])  # (15, 8)
sim_matrix   = cosine_similarity(all_vecs_mat)                 # (15, 15)

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(sim_matrix, cmap="RdYlGn", vmin=-1, vmax=1, aspect="auto")
plt.colorbar(im, ax=ax, label="Cosine similarity")

ax.set_xticks(range(len(vocab))); ax.set_yticks(range(len(vocab)))
ax.set_xticklabels(vocab, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(vocab, fontsize=8)
ax.set_title("Cosine similarity matrix (trained embeddings)\nGreen = similar, Red = dissimilar", fontsize=11)

# Annotate cells with values
for i in range(len(vocab)):
    for j in range(len(vocab)):
        ax.text(j, i, f"{sim_matrix[i, j]:.2f}", ha="center", va="center",
                fontsize=6, color="black")

plt.tight_layout()
plt.show()